In [1]:
# ======================================================
# 00. Imports & global config
# ======================================================
!pip install -q lightgbm==4.3.0 catboost==1.2.5 optuna==3.6.1 ta==0.11.0

# --------------------------------------------------
# Python imports
# --------------------------------------------------
import warnings, gc, random, math, os
import numpy as np, pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import RidgeCV
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr

from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

class CFG:
    train_path  = "/kaggle/input/drw-crypto-market-prediction/train.parquet"
    test_path   = "/kaggle/input/drw-crypto-market-prediction/test.parquet"
    sample_sub  = "/kaggle/input/drw-crypto-market-prediction/sample_submission.csv"

    target      = "label"
    seed        = 42
    n_splits    = 4
    val_size    = 20_000      # rows per fold

np.random.seed(CFG.seed)
random.seed(CFG.seed)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 16.3 MB/s eta 0:00:00


In [2]:
def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    start = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        if df[col].dtype == object:
            continue
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast="integer")
        else:
            df[col] = pd.to_numeric(df[col], downcast="float")
    end = df.memory_usage().sum() / 1024**2
    print(f"Mem ↓ {start:.1f} → {end:.1f} MB")
    return df

In [3]:
# ─────────────────────────────────────────────────────────────
#  LEAK-FREE enrich()  — robust to missing timestamp column
# ─────────────────────────────────────────────────────────────
def enrich(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    # 1) sort chronologically if the column is present
    if "timestamp" in df.columns:
        df = df.sort_values("timestamp").copy()
    else:
        df = df.copy()                       # keep original order

    # 2) base numeric features (skip 'label')
    base_cols = ["bid_qty","ask_qty","buy_qty","sell_qty","volume"] \
                + [c for c in df.columns if c.startswith("X_")]

    # 3) create safe lags / rolling stats
    for col in base_cols:
        df[f"{col}_lag1"] = df[col].shift(1)
        df[f"{col}_lag3"] = df[col].shift(3)

        roll = df[col].rolling(5, min_periods=1)
        df[f"{col}_r5_mean"] = roll.mean()
        df[f"{col}_r5_std"]  = roll.std()

    # 4) drop NaNs only in train, fill in test
    return df.dropna().reset_index(drop=True) if is_train else df.fillna(0)

In [4]:
train = pd.read_parquet(CFG.train_path)
test  = pd.read_parquet(CFG.test_path)

# add dummy asset_id for grouping consistency
train["asset_id"] = 0
test ["asset_id"] = 0

train = reduce_mem_usage(train)
test  = reduce_mem_usage(test)

train = enrich(train, is_train=True)
test  = enrich(test , is_train=False)

y      = train["label"].astype(np.float32)
X      = train.drop("label", axis=1)
X_test = test.drop("label", axis=1)

# cleanse ±inf / NaN
X      = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

Mem ↓ 3603.0 → 1806.0 MB
Mem ↓ 3682.9 → 1839.9 MB


In [5]:
tscv  = TimeSeriesSplit(n_splits=CFG.n_splits, test_size=CFG.val_size)
folds = list(tscv.split(train))

In [6]:
def lgb_pearson(y_true, y_pred):
    return "pearson", np.corrcoef(y_true, y_pred)[0,1], True

def fit_oof(model, name):
    oof   = np.zeros(len(X), dtype=np.float32)
    preds = np.zeros(len(X_test), dtype=np.float32)
    scores = []

    is_lgb = name=="LGBM"
    is_xgb = name=="XGB"

    for f,(tr,vl) in enumerate(folds):
        X_tr,y_tr = X.iloc[tr], y.iloc[tr]
        X_vl,y_vl = X.iloc[vl], y.iloc[vl]

        if is_lgb:
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_vl, y_vl)],
                eval_metric=lgb_pearson,
                callbacks=[
                    lgb.early_stopping(200, first_metric_only=True),
                    lgb.log_evaluation(50),
        ],
    )


        elif is_xgb:
            model.fit(X_tr,y_tr,
                      eval_set=[(X_vl,y_vl)],
                      eval_metric="rmse",
                      early_stopping_rounds=200)
        else:
            model.fit(X_tr,y_tr)

        oof[vl]    = model.predict(X_vl)
        preds     += model.predict(X_test) / CFG.n_splits
        score      = np.corrcoef(y_vl, oof[vl])[0,1]
        scores.append(score)
        print(f"{name} fold {f} Pearson: {score:.4f}")

    print(f"{name} CV {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    return oof, preds

In [7]:
# LightGBM
lgb_params = dict(
    objective        = "regression",   # ← handles ± sign
    learning_rate    = 0.05,
    num_leaves       = 64,
    n_estimators     = 600,
    bagging_fraction = 0.8,
    bagging_freq     = 1,
    feature_fraction = 0.9,
    random_state     = CFG.seed,
)

lgb_oof, lgb_pred = fit_oof(LGBMRegressor(**lgb_params), "LGBM")

# XGBoost
xgb_params = dict(
    objective      = "reg:squarederror",  # OK with negatives
    learning_rate  = 0.05,
    max_depth      = 6,
    subsample      = 0.8,
    colsample_bytree=0.8,
    n_estimators   = 400,
    random_state   = CFG.seed,
    missing        = np.nan,
)


xgb_oof, xgb_pred = fit_oof(XGBRegressor(**xgb_params), "XGB")


[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.535627 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 226429
[LightGBM] [Info] Number of data points in the train set: 445884, number of used features: 888
[LightGBM

In [8]:
meta_train = pd.DataFrame({"lgb": lgb_oof, "xgb": xgb_oof})
meta_test  = pd.DataFrame({"lgb": lgb_pred, "xgb": xgb_pred})

meta = RidgeCV(alphas=[0.01,0.1,1.0,10.0], cv=5)
meta.fit(meta_train, y)

stack_pred = meta.predict(meta_test)
print("Stack train Pearson:", pearsonr(y, meta.predict(meta_train))[0])

Stack train Pearson: 0.051509194


In [9]:
from scipy.stats import rankdata, norm

def rank_gauss(arr):
    r = rankdata(arr) / (len(arr) + 1)
    return norm.ppf(r)

stack_pred = rank_gauss(stack_pred)

sub = pd.read_csv(CFG.sample_sub)
sub["prediction"] = stack_pred.astype(np.float32)
sub.to_csv("submission_fixed.csv", index=False)
sub.head()


,ID,prediction
0,1,1.314501
1,2,0.254763
2,3,0.118263
3,4,-0.462842
4,5,1.415016
